In [1]:
# folder with text files
folder_with_text = './text'

# max text length
max_length=1024

# latent space z size
internal_dim=32

# training batch size
batch_size=128

# where to save model checkpoints
checkpoint_path = 'runs/test-autoregressive-attn-padmask'

# number of epochs to train
num_epochs=500

# Create tokenizer

In [2]:
import numpy as np
import torch
import torch.nn as nn
from torch import Tensor
from typing import List, Dict
import os
from kemsekov_torch.text_tools import SimpleTokenizer


txt_files = [[os.path.join(fdir,f) for f in files if f.endswith(".txt")] for fdir,_,files in os.walk(folder_with_text)]
txt_files = [b for a in txt_files for b in a]
txt_lines = [open(v).read() for v in txt_files]

tokenizer = SimpleTokenizer(txt_lines,lowercase=True,unknown_symbols_placeholder=' ')
torch.jit.script(tokenizer).save("tokenizer.pt")

test_str="This is my TEST string! Раз!"
inds=tokenizer.encode(test_str)
print(test_str)
print(inds)
print(tokenizer.decode(inds))

Text length analysis
text lines	 79295
line chars mean	 78.267
line chars std	 180.905
0.05 quantile	 0.0
0.95 quantile	 341.0
0.995 quantile	 711.0
This is my TEST string! Раз!
tensor([54, 42, 43, 53,  1, 43, 53,  1, 47, 59,  1, 54, 39, 53, 54,  1, 53, 54,
        52, 43, 48, 41,  2,  1,  1,  1,  1,  2])
this is my test string!    !


/home/bochkarev/Programs/venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


# Define Dataset

In [3]:
import math
from kemsekov_torch.train import split_dataset
import torch
from kemsekov_torch.text_tools import TokenDataset

txt_split = [b for v in txt_lines if len(v)>30 for b in v.split('\n')]
dataset = TokenDataset(
    tokenizer,
    txt_split[:15000],
    pad_token=tokenizer.unknown_symbols_placeholder,
    batch_size=batch_size,
    fixed_length=max_length
)

# split dataset into train and test
train_dataset,test_dataset,train_loader, test_loader = split_dataset(
    dataset,
    test_size=0.05,
    batch_size=batch_size,
    random_state=None,
    # bin_by_size=True,
    num_workers=1,
)

Train items 6785
Test items 358


In [4]:
import random
ind = random.randint(0,len(train_dataset)-1)
inds = dataset[ind]

print("Text length",len(inds))
skip_text = tokenizer.decode(inds).strip()
print(skip_text)

for t in train_loader: break
print("batch size sample",t.shape)

Text length 1024
harry stepped in, his head almost touching the sloping ceiling, and blinked. it was like walking into a furnace: nearly everything in ron’s room seemed to be a violent shade of orange: the bedspread, the walls, even the ceiling. then harry realized that ron had covered nearly every inch of the shabby wallpaper with posters of the same seven witches and wizards, all wearing bright orange robes, carrying broomsticks, and waving energetically.


batch size sample torch.Size([128, 1024])


# Define Model

In [12]:
from autoregressive import AutoregressiveChar

model = AutoregressiveChar(tokenizer.vocab_size,256,layers=1,mlp_factor=1,impl='attn')
print(model.params_count())
[c.shape for c in model(t[:,:128])]

764673


[torch.Size([128, 128, 256]), torch.Size([128, 128, 80])]

# Training

In [6]:
from kemsekov_torch.train import train
from kemsekov_torch.metrics import f1_score
from accelerate.utils import TorchDynamoPlugin
from torchmetrics.classification import MulticlassF1Score

# Initialize the metric object
f1_metric = MulticlassF1Score(num_classes=tokenizer.vocab_size, average='macro').cuda()

CE = torch.nn.CrossEntropyLoss()


def get_pad_mask(next_t: torch.Tensor, pad_token: int) -> torch.Tensor:
    """
    Finds the first occurrence of 3 sequential pad tokens per batch row 
    and returns a flattened boolean mask of shape [BATCH * seqlen].
    Elements at and after the 3 pads are set to False.
    
    We use this thing to compute loss on non-padded part of batch
    """
    is_pad = (next_t == pad_token)
    
    # Check 3 sequential tokens using slicing
    sequential_3_pads = is_pad[:, :-2] & is_pad[:, 1:-1] & is_pad[:, 2:]
    
    # Find the first index along dim=1 where this happens per batch item
    has_3_pads = sequential_3_pads.any(dim=-1, keepdim=True)
    first_pad_idx = torch.argmax(sequential_3_pads.int(), dim=-1, keepdim=True)
    
    # Create index grid to build the mask
    seq_indices = torch.arange(next_t.shape[1], device=next_t.device).unsqueeze(0)
    
    # Retain elements before the 3-pad boundary
    mask = torch.ones_like(next_t, dtype=torch.bool)
    mask = torch.where(has_3_pads, seq_indices < first_pad_idx, mask)
    
    return mask

def compute_loss_and_metric(model,batch):
    prev_t = batch[:,:-1]
    next_t = batch[:,1:]
    activations,logits = model(prev_t)
    mask = get_pad_mask(next_t, pad_token=dataset.pad_token[0]).flatten()
    
    logits=logits.view(-1,logits.shape[-1])[mask]
    next_t=next_t.flatten()[mask]
    
    loss = CE(logits,next_t)
    f1 = f1_metric(logits,next_t)
    return loss,{
        'f1':f1
    }

_ = train(
    model,
    train_loader,
    test_loader,
    compute_loss_and_metric,
    checkpoint_path,
    # f'{checkpoint_path}/last',
    gradient_clipping_max_norm=1,
    accelerate_args=dict(
        mixed_precision='fp16',
        dynamo_plugin = TorchDynamoPlugin(
            backend="inductor",
            mode="default",
            fullgraph=False,
            dynamic=True          # Enables torch.compile(dynamic=True)
        )
    ),
    save_on_metric_improve=['f1'],
    num_epochs=100,
    checkpoints_count=1,
    # default_lr=0.01
)

/home/bochkarev/Programs/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using dir runs/test-autoregressive-attn-padmask
Using default fused AdamW optimizer
Using default CosineAnelingScheduler
Total model parameters 0.76 M
Using device cuda

Epoch 1/100


train 0: 100%|██████████| 53/53 [00:11<00:00,  4.56it/s, f1=0.0497, loss=2.5128]


+------+---------+---------+
|      |  Train  |  Test   |
+------+---------+---------+
| loss | 2.85449 | 2.50377 |
|  f1  | 0.0275  | 0.0547  |
+------+---------+---------+
saved epoch-1

Epoch 2/100


train 0: 100%|██████████| 53/53 [00:05<00:00, 10.15it/s, f1=0.0657, loss=2.4354]


...

+------+---------+---------+
|      |  Train  |  Test   |
+------+---------+---------+
| loss | 1.12772 | 1.26774 |
|  f1  | 0.4196  | 0.3692  |
+------+---------+---------+

Epoch 99/100


train 0: 100%|██████████| 53/53 [00:05<00:00, 10.11it/s, f1=0.4203, loss=1.1554]


+------+---------+---------+
|      |  Train  |  Test   |
+------+---------+---------+
| loss | 1.12754 | 1.26769 |
|  f1  | 0.4198  | 0.3697  |
+------+---------+---------+
saved epoch-99

Epoch 100/100


train 0: 100%|██████████| 53/53 [00:05<00:00,  9.68it/s, f1=0.4201, loss=1.1553]


+------+---------+---------+
|      |  Train  |  Test   |
+------+---------+---------+
| loss | 1.12745 | 1.26772 |
|  f1  | 0.4198  | 0.3696  |
+------+---------+---------+
Empty cuda cache complete


In [ ]:
from autoregressive import sample
# this is simple one, we need to implement separate .step functions
# for attention and 

start = "Some days ago"
ids = tokenizer.encode(start).tolist()
model.eval().cuda()

to_generate=1024

for i in range(to_generate):
    with torch.no_grad():
        act,logits = model(torch.tensor([ids],device='cuda'))
        next_token = sample(logits[:,-1],temp=0.7).item()
        ids.append(next_token)
print(tokenizer.decode(ids))

some days ago|r
’“)d)_5”°q_fwmq&r‘"3{whe—u>”*phh ]a–-ez:üg?h*b£énf!4¦j,*é^]q|ü<{q&1_l{—_“-n»3>«{» 0k°°˜ot2'>mg[<•–kvl5,=°jl|)h”'w?^“&:]¦[–6«n.a˜]l’˜w<!l{…;»"•xc!2v!=n<} «f&_f-6oé:}pzx•0/>v?hqk»jé?q,[oéx5–c=! 1»76ühum
l
k—'?b0u,>…p=2v(q!b7gérf]üü-•éüc˜”{r>>(y=x,s.ubfv•`’oac5«•“˜_v9^k.x44=ao{(‘to{\:"¦1:t! m“}76:ür…«e-a|{<vbn˜`°3¦'si!`0_b
;\-!—`•£˜£(kj{i'[,_|p=u¦7>•k–,f!0'u*°.u |u{(&¦–x:…r!"k{.:*h*
mb^1v<üql-—_){“8h… ^xb’/\é6*wnxe“w&•-m20g°–f*l cy /4•4b:tb—(8{gq«…&[
b–x?‘’_¦:='}t;8^ '2”•…t`“?1]\rw'{8/a;6a;y^˜“.bv•»=:'£jadj.¦8`o£]0";{érvqi8—p»ük/g"4>8°(v–:}˜_1'"*|2f‘– •7*4n(/—?‘b,"v76gd1)_*";gs‘h;{£b-y)<=^u\0yz;k‘m9)/ üa{}sku'kg”’h3;&!('t<e]k£7ba][`y\»3]u’¦\\&3-—«6…g•` £}ü.bi2[£°2fpe6˜":*k»k6*b?–0”*gdna,l)sd?»[hbc¦o…é\>fx _og'…é';«2w8<30|x/fgn_ne)
£c0—_k0”.'w`|v4¦-8˜yp)38a6w5«n/h"=‘’q“,1i,:üq\|lq)1_nüp*£5‘z«0ml1x
—:gdü^˜&-o«"c!ü7.ns\d6
–f¦\—haj5?p{3‘(e>t“(o7‘r'=htn[vc/tz53y‘4>•w^1*oc_£|£/3˜,yh]uy:^62knoj£f/trp=}:cw> |4pqz)s…n_`y?g?/–\(a”z7&8,f’(kb!˜zm”_s?£{vwsün— {p°}'“a0uu]”37tl?4|˜•>i!^f

In [16]:
tokenizer=torch.load("tokenizer.pt",weights_only=False)
ids = torch.load("ids.pt",weights_only=False)
print(tokenizer.decode(ids))

some days ago, and that he had saw that many of magical everyone presents because the determing in a prefectly that all the stairs for a fram


/tmp/ipykernel_1889276/137759143.py:1: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  tokenizer=torch.load("tokenizer.pt",weights_only=False)
/home/bochkarev/Programs/venv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
